# 🇮🇳 Economic Indicator Intelligence Dashboard — India
### Phase 2: Analysis & Storytelling

**What this notebook does:**
- Analyses India's key macroeconomic indicators from 2000 to 2025
- Identifies major economic shocks and explains what caused them
- Compares India against 5 major economies
- Translates every chart into plain English

**Data sources:** FRED (RBI repo rate, CPI, USD/INR, Industrial Production) + World Bank (GDP, unemployment, trade, investment)

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# ── Chart style ───────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  '#0f1117',
    'axes.facecolor':    '#0f1117',
    'axes.edgecolor':    '#2a2d3a',
    'axes.labelcolor':   '#c9d1d9',
    'axes.grid':         True,
    'grid.color':        '#21262d',
    'grid.linestyle':    '--',
    'grid.linewidth':    0.6,
    'xtick.color':       '#8b949e',
    'ytick.color':       '#8b949e',
    'text.color':        '#c9d1d9',
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'legend.facecolor':  '#161b22',
    'legend.edgecolor':  '#30363d',
    'figure.dpi':        130,
})

# ── Colour palette ────────────────────────────────────────────────
C = {
    'india':      '#f97316',   # saffron orange  — India's primary colour
    'us':         '#58a6ff',   # blue
    'china':      '#f85149',   # red
    'germany':    '#3fb950',   # green
    'uk':         '#a371f7',   # purple
    'brazil':     '#ffa657',   # amber
    'accent':     '#ff7b72',   # shock annotation
    'soft':       '#8b949e',   # secondary text
    'repo':       '#79c0ff',   # repo rate line
    'inr':        '#d2a8ff',   # rupee line
    'ip':         '#56d364',   # industrial production
}

COUNTRY_COLORS = {
    'India':          C['india'],
    'United States':  C['us'],
    'China':          C['china'],
    'Germany':        C['germany'],
    'United Kingdom': C['uk'],
    'Brazil':         C['brazil'],
}

print('✅ Setup complete')

## 1. Load Data

In [ ]:
# ── Load FRED (India, high-frequency) ────────────────────────────
fred = pd.read_csv('data/india_fred_all.csv', parse_dates=['date'])

inflation  = fred[fred['indicator'] == 'inflation_yoy_pct'].copy()
repo       = fred[fred['indicator'] == 'repo_rate'].copy()
usd_inr    = fred[fred['indicator'] == 'usd_inr'].copy()
ind_prod   = fred[fred['indicator'] == 'industrial_production'].copy()

# Normalise industrial production to index (2015 = 100) for readability
base = ind_prod[ind_prod['date'].dt.year == 2015]['value'].mean()
ind_prod['value'] = (ind_prod['value'] / base) * 100

# ── Load World Bank (multi-country, annual) ───────────────────────
wb = pd.read_csv('data/worldbank_all_countries.csv', parse_dates=['date'])
wb['year'] = wb['date'].dt.year

gdp_growth     = wb[wb['indicator'] == 'gdp_growth_pct']
gdp_per_capita = wb[wb['indicator'] == 'gdp_per_capita']
wb_inflation   = wb[wb['indicator'] == 'inflation_pct']
unemployment   = wb[wb['indicator'] == 'unemployment_pct']
current_acct   = wb[wb['indicator'] == 'current_account_gdp']

print('✅ Data loaded')
print(f'   FRED indicators : {fred["indicator"].unique().tolist()}')
print(f'   WB indicators   : {wb["indicator"].unique().tolist()}')
print(f'   WB countries    : {wb["country"].unique().tolist()}')

## 2. Helper Functions

In [ ]:
# ── Shock annotation helper ───────────────────────────────────────
# Call this on any axis to mark known economic shocks as vertical bands

SHOCKS = {
    'GFC\n(2008)':      ('2008-09-01', '2009-06-01'),
    'Taper\nTantrum':   ('2013-05-01', '2013-09-01'),
    'Demonetisation':   ('2016-11-01', '2017-03-01'),
    'COVID-19\n(2020)': ('2020-03-01', '2020-09-01'),
    'Inflation\nSurge': ('2021-10-01', '2022-12-01'),
}

def add_shocks(ax, alpha=0.12, label=True):
    """Shade shock periods and optionally label them."""
    for name, (start, end) in SHOCKS.items():
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
                   color=C['accent'], alpha=alpha, zorder=0)
        if label:
            mid = pd.Timestamp(start) + (pd.Timestamp(end) - pd.Timestamp(start)) / 2
            ymin, ymax = ax.get_ylim()
            ax.text(mid, ymax * 0.97, name, ha='center', va='top',
                    fontsize=7.5, color=C['accent'], alpha=0.85,
                    rotation=0, linespacing=1.3)

def insight_box(text):
    """Print a formatted plain-English insight beneath a chart."""
    border = '─' * 72
    print(f'\n💡 INSIGHT\n{border}')
    # Word-wrap to 72 chars
    import textwrap
    for line in textwrap.wrap(text, width=72):
        print(f'   {line}')
    print(border)

print('✅ Helpers ready')

---
# Section 1 — India Macro Overview
### The full picture: 25 years of India's economy in four charts

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 18), sharex=False)
fig.suptitle('India — Macroeconomic Indicators (2000–2025)',
             fontsize=16, fontweight='bold', color='white', y=0.98)

# ── 1. Inflation ──────────────────────────────────────────────────
ax = axes[0]
ax.plot(inflation['date'], inflation['value'],
        color=C['india'], linewidth=1.8, label='YoY Inflation %')
ax.axhline(4, color=C['soft'], linestyle=':', linewidth=1,
           label='RBI target (4%)')
ax.axhline(6, color=C['accent'], linestyle=':', linewidth=1,
           alpha=0.6, label='Upper tolerance (6%)')
ax.fill_between(inflation['date'], inflation['value'], 4,
                where=inflation['value'] > 4,
                color=C['accent'], alpha=0.12, label='Above target')
ax.set_ylabel('Inflation (%)', color=C['soft'])
ax.set_title('CPI Inflation — Year on Year', color='white', pad=8)
ax.legend(fontsize=8, loc='upper right')
add_shocks(ax)

# ── 2. Repo Rate ──────────────────────────────────────────────────
ax = axes[1]
ax.step(repo['date'], repo['value'],
        color=C['repo'], linewidth=2, where='post', label='RBI Repo Rate %')
ax.fill_between(repo['date'], repo['value'], step='post',
                color=C['repo'], alpha=0.12)
ax.set_ylabel('Repo Rate (%)', color=C['soft'])
ax.set_title('RBI Repo Rate — Monetary Policy Stance', color='white', pad=8)
ax.legend(fontsize=8, loc='upper right')
add_shocks(ax)

# ── 3. USD/INR ────────────────────────────────────────────────────
ax = axes[2]
ax.plot(usd_inr['date'], usd_inr['value'],
        color=C['inr'], linewidth=1.6, label='₹ per $1 USD')
ax.set_ylabel('INR per USD', color=C['soft'])
ax.set_title('USD/INR Exchange Rate — Rupee Depreciation Story', color='white', pad=8)
ax.legend(fontsize=8, loc='upper left')
ax.invert_yaxis()   # lower = stronger rupee; makes the trend intuitive
ax.set_ylabel('₹ per $1 (inverted: up = stronger ₹)', color=C['soft'])
add_shocks(ax)

# ── 4. Industrial Production ──────────────────────────────────────
ax = axes[3]
ax.plot(ind_prod['date'], ind_prod['value'],
        color=C['ip'], linewidth=1.4, alpha=0.8, label='IIP (2015=100)')
# 12-month rolling average for trend clarity
ind_prod_sorted = ind_prod.sort_values('date')
rolling = ind_prod_sorted['value'].rolling(12).mean()
ax.plot(ind_prod_sorted['date'], rolling,
        color='white', linewidth=2, label='12-month trend', alpha=0.7)
ax.set_ylabel('Index (2015 = 100)', color=C['soft'])
ax.set_title('Index of Industrial Production — Economic Activity Pulse', color='white', pad=8)
ax.legend(fontsize=8, loc='upper left')
add_shocks(ax)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('data/chart_01_india_overview.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()
print('Chart saved.')

In [ ]:
insight_box("""
India's macroeconomic story from 2000 to 2025 is one of remarkable growth interrupted
by three distinct shocks. Inflation, which averaged around 7% in the 2000s, was
brought under a formal target of 4% (±2%) only in 2016 when the RBI adopted flexible
inflation targeting — a structural shift visible in how much calmer inflation has been
since then. The rupee has followed a clear long-run depreciation trend, falling from
around ₹45 to ₹83 per dollar, largely reflecting India's higher inflation relative to
the US rather than economic weakness. Industrial production shows a sharp collapse in
April 2020 during the COVID-19 lockdown — the steepest single-month fall in modern
Indian economic history — followed by a strong V-shaped recovery.
""".strip())

---
# Section 2 — The RBI Playbook
### How India's central bank responds to inflation

In [ ]:
# Align inflation and repo rate on a common monthly date index
inf_m = inflation.set_index('date')['value'].resample('MS').mean()
repo_m = repo.set_index('date')['value'].resample('MS').ffill()  # forward-fill quarterly to monthly

combined = pd.DataFrame({'inflation': inf_m, 'repo': repo_m}).dropna()

fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(combined.index, combined['inflation'],
         color=C['india'], linewidth=2, label='Inflation (YoY %)')
ax1.axhline(4, color=C['soft'], linestyle=':', linewidth=1, alpha=0.7)
ax1.axhline(6, color=C['accent'], linestyle=':', linewidth=1, alpha=0.5)
ax1.set_ylabel('Inflation (%)', color=C['india'])
ax1.tick_params(axis='y', labelcolor=C['india'])
ax1.set_ylim(-2, 16)

ax2 = ax1.twinx()
ax2.step(combined.index, combined['repo'],
         color=C['repo'], linewidth=2.5, where='post',
         label='Repo Rate (%)', alpha=0.9)
ax2.set_ylabel('Repo Rate (%)', color=C['repo'])
ax2.tick_params(axis='y', labelcolor=C['repo'])
ax2.set_ylim(2, 12)

# Annotate key RBI decisions
annotations = [
    ('2008-10-01', 9.0,  'RBI cuts\naggressively'),
    ('2010-06-01', 7.5,  'Hiking cycle\nbegins'),
    ('2020-04-01', 4.4,  'Emergency\ncut to 4.4%'),
    ('2022-05-01', 4.0,  'Hikes restart'),
]
for date, y, label in annotations:
    ax2.annotate(label,
                 xy=(pd.Timestamp(date), y),
                 xytext=(pd.Timestamp(date), y + 1.8),
                 fontsize=8, color='white', ha='center',
                 arrowprops=dict(arrowstyle='->', color=C['soft'], lw=1),
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='#161b22',
                           edgecolor=C['soft'], alpha=0.9))

add_shocks(ax1, label=True)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

ax1.set_title('The RBI Playbook — Inflation vs Repo Rate (2000–2024)',
              color='white', fontsize=13, pad=10)
ax1.set_xlabel('')

plt.tight_layout()
plt.savefig('data/chart_02_rbi_playbook.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()

In [ ]:
# Compute average lag between inflation peak and repo rate peak
# Using a simple cross-correlation approach
inf_vals  = combined['inflation'].values
repo_vals = combined['repo'].values
corr      = np.correlate(inf_vals - inf_vals.mean(),
                          repo_vals - repo_vals.mean(), mode='full')
lags      = np.arange(-(len(combined)-1), len(combined))
peak_lag  = lags[np.argmax(corr)]

print(f'Cross-correlation peak lag: {peak_lag} months')
print('(Positive = repo rate moves after inflation)')

insight_box(f"""
The RBI does not react to inflation instantly — it watches trends, waits for data
confirmation, and then moves. Cross-correlation analysis shows the repo rate tends
to follow inflation by approximately {abs(peak_lag)} months on average. This lag is
intentional: raising rates too quickly can choke growth, while waiting too long lets
inflation become entrenched. The 2010–11 tightening cycle and the 2022 post-COVID
hikes both illustrate this pattern clearly — inflation rose first, RBI watched, then
hiked decisively. Conversely, during COVID in 2020, the RBI cut the repo rate to a
historic low of 4.4% even as supply-side inflation was rising — a calculated bet
that growth support mattered more than inflation control in a once-in-a-century crisis.
""".strip())

---
# Section 3 — India vs The World
### How does India compare to major economies?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('India vs The World — Key Macro Indicators (2000–2024)',
             fontsize=15, fontweight='bold', color='white', y=1.01)

SHOCK_YEARS = [2008, 2009, 2020]

def plot_country_lines(ax, df, indicator, title, ylabel, invert=False):
    data = df[df['indicator'] == indicator]
    for country, color in COUNTRY_COLORS.items():
        sub = data[data['country'] == country].sort_values('year')
        if sub.empty:
            continue
        lw = 3 if country == 'India' else 1.4
        alpha = 1.0 if country == 'India' else 0.65
        ax.plot(sub['year'], sub['value'],
                color=color, linewidth=lw, alpha=alpha, label=country)
    for yr in SHOCK_YEARS:
        ax.axvline(yr, color=C['accent'], linestyle='--',
                   linewidth=0.8, alpha=0.5)
    ax.set_title(title, color='white', pad=8, fontsize=11)
    ax.set_ylabel(ylabel, color=C['soft'])
    if invert:
        ax.invert_yaxis()
    ax.legend(fontsize=7.5, loc='best', ncol=2)

plot_country_lines(axes[0,0], wb, 'gdp_growth_pct',
                   'GDP Growth Rate (%)', 'Annual Growth (%)')

plot_country_lines(axes[0,1], wb, 'inflation_pct',
                   'Inflation Rate (%)', 'Annual Inflation (%)')

plot_country_lines(axes[1,0], wb, 'unemployment_pct',
                   'Unemployment Rate (%)', '% of Labour Force')

plot_country_lines(axes[1,1], wb, 'current_account_gdp',
                   'Current Account Balance (% of GDP)', '% of GDP')

# Add vertical line labels on first chart only
for yr, label in [(2008, 'GFC'), (2020, 'COVID')]:
    axes[0,0].text(yr + 0.2, axes[0,0].get_ylim()[0] * 0.85,
                   label, fontsize=8, color=C['accent'], alpha=0.8)

plt.tight_layout()
plt.savefig('data/chart_03_india_vs_world.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()

In [ ]:
# Compute India's average GDP growth vs peer average
india_gdp  = gdp_growth[gdp_growth['country'] == 'India']['value'].mean()
others_gdp = gdp_growth[gdp_growth['country'] != 'India']['value'].mean()
india_inf  = wb_inflation[wb_inflation['country'] == 'India']['value'].mean()

print(f"India avg GDP growth (2000–2024): {india_gdp:.1f}%")
print(f"Peer avg GDP growth  (2000–2024): {others_gdp:.1f}%")
print(f"India avg inflation  (2000–2024): {india_inf:.1f}%")

insight_box(f"""
India has been the fastest-growing large economy over the past 25 years, averaging
{india_gdp:.1f}% GDP growth compared to {others_gdp:.1f}% for the peer group. Even during
the 2008 Global Financial Crisis — when the US, Germany and UK all contracted —
India grew at 3.9%, a testament to its domestically-driven economy that was less
exposed to Western financial markets. The one clear exception is COVID-19 in 2020,
where India's strict lockdown caused a 6.6% contraction — steeper than most peers
because of the economy's large informal sector that had no social safety net to fall
back on. On inflation, India's {india_inf:.1f}% average is higher than developed
economies but lower than Brazil, reflecting the classic emerging-market tradeoff:
fast growth tends to generate more inflation, especially when food prices — which
carry a higher weight in India's CPI basket — are volatile.
""".strip())

---
# Section 4 — Shock Analysis
### What actually happened during India's two biggest crises?

In [ ]:
# ── Helper: extract a window around a shock ───────────────────────
def shock_window(df, date_col, start, end):
    mask = (df[date_col] >= start) & (df[date_col] <= end)
    return df[mask].copy()

# ── SHOCK 1: Global Financial Crisis 2008 ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Shock Analysis 1: Global Financial Crisis (2007–2010)',
             fontsize=13, fontweight='bold', color='white')

gfc_start, gfc_end = '2006-01-01', '2011-01-01'

# Inflation during GFC
ax = axes[0]
d = shock_window(inflation, 'date', gfc_start, gfc_end)
ax.plot(d['date'], d['value'], color=C['india'], linewidth=2)
ax.axvspan('2008-09-01', '2009-06-01', color=C['accent'], alpha=0.15)
ax.axvline(pd.Timestamp('2008-09-15'), color=C['accent'],
           linestyle='--', linewidth=1.2, label='Lehman collapse')
ax.set_title('Inflation (%)', color='white')
ax.legend(fontsize=8)

# Repo rate during GFC
ax = axes[1]
d = shock_window(repo, 'date', gfc_start, gfc_end)
ax.step(d['date'], d['value'], color=C['repo'], linewidth=2.5, where='post')
ax.axvspan('2008-09-01', '2009-06-01', color=C['accent'], alpha=0.15)
ax.set_title('RBI Repo Rate (%)', color='white')

# GDP growth comparison during GFC (World Bank annual)
ax = axes[2]
gfc_years = [2007, 2008, 2009, 2010]
for country, color in COUNTRY_COLORS.items():
    sub = gdp_growth[(gdp_growth['country'] == country) &
                     (gdp_growth['year'].isin(gfc_years))].sort_values('year')
    if sub.empty: continue
    lw = 3 if country == 'India' else 1.4
    ax.plot(sub['year'], sub['value'], color=color,
            linewidth=lw, marker='o', markersize=4, label=country)
ax.axhline(0, color='white', linestyle='-', linewidth=0.6, alpha=0.4)
ax.set_title('GDP Growth — GFC Years', color='white')
ax.legend(fontsize=7.5, ncol=2)
ax.set_xticks(gfc_years)

for ax in axes:
    ax.set_facecolor('#0f1117')

plt.tight_layout()
plt.savefig('data/chart_04a_gfc_shock.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()

In [ ]:
india_2008 = float(gdp_growth[(gdp_growth['country']=='India') &
                               (gdp_growth['year']==2009)]['value'].values[0])
us_2008    = float(gdp_growth[(gdp_growth['country']=='United States') &
                               (gdp_growth['year']==2009)]['value'].values[0])

print(f"India GDP growth 2009: {india_2008:.1f}%")
print(f"US GDP growth    2009: {us_2008:.1f}%")

insight_box(f"""
The 2008 Global Financial Crisis barely touched India's growth rate. While the United
States contracted by {abs(us_2008):.1f}% in 2009 and Europe entered a deep recession,
India still grew at {india_2008:.1f}%. The reason: India's banks had very limited
exposure to the toxic US mortgage-backed securities that caused the crisis. The
RBI's conservative banking regulations — often criticised as too cautious in good
times — turned out to be exactly the right shield. Where India did feel the crisis
was through trade: exports fell sharply as Western consumers stopped spending, and
foreign investors pulled money out of emerging markets, weakening the rupee. The RBI
responded by cutting the repo rate from 9% to 4.75% between October 2008 and April
2009 — one of its fastest easing cycles on record.
""".strip())

In [ ]:
# ── SHOCK 2: COVID-19 2020 ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Shock Analysis 2: COVID-19 Pandemic (2019–2022)',
             fontsize=13, fontweight='bold', color='white')

covid_start, covid_end = '2019-01-01', '2023-01-01'

# Industrial production — the sharpest COVID signal
ax = axes[0]
d = shock_window(ind_prod, 'date', covid_start, covid_end)
ax.plot(d['date'], d['value'], color=C['ip'], linewidth=1.8)
ax.axvspan('2020-03-01', '2020-09-01', color=C['accent'], alpha=0.2)
ax.annotate('Lockdown\nApril 2020',
             xy=(pd.Timestamp('2020-04-01'),
                 d[d['date'] == d['date'].min()]['value'].values[0] if len(d) else 40),
             xytext=(pd.Timestamp('2020-08-01'), 75),
             fontsize=8, color=C['accent'],
             arrowprops=dict(arrowstyle='->', color=C['accent']))
ax.set_title('Industrial Production (2015=100)', color='white')

# Inflation during COVID
ax = axes[1]
d = shock_window(inflation, 'date', covid_start, covid_end)
ax.plot(d['date'], d['value'], color=C['india'], linewidth=2)
ax.axhline(6, color=C['accent'], linestyle=':', linewidth=1, alpha=0.7,
           label='Upper tolerance (6%)')
ax.axvspan('2020-03-01', '2020-09-01', color=C['accent'], alpha=0.15)
ax.set_title('Inflation (%) — Supply Shock', color='white')
ax.legend(fontsize=8)

# GDP growth comparison during COVID
ax = axes[2]
covid_years = [2019, 2020, 2021, 2022]
for country, color in COUNTRY_COLORS.items():
    sub = gdp_growth[(gdp_growth['country'] == country) &
                     (gdp_growth['year'].isin(covid_years))].sort_values('year')
    if sub.empty: continue
    lw = 3 if country == 'India' else 1.4
    ax.plot(sub['year'], sub['value'], color=color,
            linewidth=lw, marker='o', markersize=4, label=country)
ax.axhline(0, color='white', linestyle='-', linewidth=0.6, alpha=0.4)
ax.set_title('GDP Growth — COVID Years', color='white')
ax.legend(fontsize=7.5, ncol=2)
ax.set_xticks(covid_years)

for ax in axes:
    ax.set_facecolor('#0f1117')

plt.tight_layout()
plt.savefig('data/chart_04b_covid_shock.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()

In [ ]:
india_2020 = float(gdp_growth[(gdp_growth['country']=='India') &
                               (gdp_growth['year']==2020)]['value'].values[0])
india_2021 = float(gdp_growth[(gdp_growth['country']=='India') &
                               (gdp_growth['year']==2021)]['value'].values[0])

# Peak COVID inflation
covid_inf = inflation[(inflation['date'] >= '2020-01-01') &
                      (inflation['date'] <= '2021-06-01')]['value'].max()

print(f"India GDP 2020: {india_2020:.1f}%, 2021 rebound: {india_2021:.1f}%")
print(f"Peak inflation during COVID window: {covid_inf:.1f}%")

insight_box(f"""
COVID-19 hit India harder than the 2008 crisis — GDP contracted {abs(india_2020):.1f}% in
2020, the worst performance since Independence. The April 2020 lockdown was one of
the strictest in the world, and industrial production collapsed by nearly 60% in that
single month. Counterintuitively, inflation rose during the lockdown (peaking near
{covid_inf:.1f}%) rather than falling — this is a supply shock story, not a demand one.
Factories shut down, supply chains broke, and food prices spiked even as people
earned less. This put the RBI in an impossible position: raise rates to fight
inflation, or cut rates to support growth? It chose growth, cutting the repo rate to
a historic low of 4%. The gamble paid off — India rebounded {india_2021:.1f}% in 2021,
the fastest recovery among all major economies, driven by pent-up demand, government
infrastructure spending, and a faster-than-expected vaccine rollout.
""".strip())

---
# Section 5 — India's Development Story
### GDP per capita: how much richer has the average Indian become?

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for country, color in COUNTRY_COLORS.items():
    sub = gdp_per_capita[gdp_per_capita['country'] == country].sort_values('year')
    if sub.empty: continue
    lw    = 3 if country == 'India' else 1.6
    alpha = 1.0 if country == 'India' else 0.6
    ax.plot(sub['year'], sub['value'], color=color,
            linewidth=lw, alpha=alpha, label=country)
    # Label the endpoint
    last = sub.iloc[-1]
    ax.text(last['year'] + 0.3, last['value'], f"${last['value']:,.0f}",
            fontsize=8, color=color, va='center')

for yr in [2008, 2020]:
    ax.axvline(yr, color=C['accent'], linestyle='--',
               linewidth=0.8, alpha=0.5)

ax.set_title('GDP per Capita (Constant 2015 USD) — The Development Gap',
             color='white', fontsize=13, pad=10)
ax.set_ylabel('USD per person (2015 prices)', color=C['soft'])
ax.legend(fontsize=9, loc='upper left')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('data/chart_05_gdp_per_capita.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()

In [ ]:
india_pc_2000 = float(gdp_per_capita[(gdp_per_capita['country']=='India') &
                                      (gdp_per_capita['year']==2000)]['value'].values[0])
india_pc_last = gdp_per_capita[gdp_per_capita['country']=='India'].sort_values('year').iloc[-1]
us_pc_last    = gdp_per_capita[gdp_per_capita['country']=='United States'].sort_values('year').iloc[-1]
multiple      = india_pc_last['value'] / india_pc_2000

print(f"India GDP/capita 2000: ${india_pc_2000:,.0f}")
print(f"India GDP/capita {india_pc_last['year']:.0f}: ${india_pc_last['value']:,.0f}")
print(f"Increase: {multiple:.1f}x")
print(f"US GDP/capita {us_pc_last['year']:.0f}: ${us_pc_last['value']:,.0f}")

insight_box(f"""
Perhaps the most powerful number in this entire analysis: India's GDP per capita
has grown {multiple:.1f}x since 2000 — from ${india_pc_2000:,.0f} to
${india_pc_last['value']:,.0f} per person. That represents hundreds of millions of
people moving out of poverty within a single generation. Yet the chart also shows
how much runway remains — the US GDP per capita of ${us_pc_last['value']:,.0f} is
roughly {us_pc_last['value']/india_pc_last['value']:.0f}x India's. This gap is both a
challenge and an opportunity: it means India has decades of catch-up growth ahead,
driven by demographics (the youngest large workforce in the world),
urbanisation, and digital infrastructure. The question is not whether India will
grow, but whether it can grow fast enough to absorb 10 million new workers
entering the labour market every year.
""".strip())

---
# Section 6 — The Rupee Story
### Why has the rupee depreciated, and should we worry?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('The Rupee Depreciation Story',
             fontsize=13, fontweight='bold', color='white')

# ── Left: Raw USD/INR ─────────────────────────────────────────────
ax = axes[0]
ax.plot(usd_inr['date'], usd_inr['value'],
        color=C['inr'], linewidth=1.8)
ax.set_title('USD/INR Exchange Rate', color='white')
ax.set_ylabel('₹ per $1 USD', color=C['soft'])
shocks_inr = [
    ('2008-09-01', 'GFC'),
    ('2013-05-01', 'Taper\nTantrum'),
    ('2020-03-01', 'COVID'),
    ('2022-02-01', 'Russia-\nUkraine'),
]
for date, label in shocks_inr:
    ax.axvline(pd.Timestamp(date), color=C['accent'],
               linestyle='--', linewidth=1, alpha=0.6)
    y_pos = usd_inr['value'].max() * 0.92
    ax.text(pd.Timestamp(date), y_pos, label,
            fontsize=7.5, color=C['accent'], ha='center')

# ── Right: Inflation differential (India minus US) ────────────────
# Theory: currencies depreciate at roughly the rate of inflation differential
ax = axes[1]
us_inf = wb_inflation[wb_inflation['country']=='United States'].set_index('year')['value']
in_inf = wb_inflation[wb_inflation['country']=='India'].set_index('year')['value']
inf_diff = (in_inf - us_inf).dropna()

colors_bar = [C['accent'] if v > 0 else C['ip'] for v in inf_diff.values]
ax.bar(inf_diff.index, inf_diff.values, color=colors_bar, alpha=0.8, width=0.7)
ax.axhline(0, color='white', linewidth=0.8, alpha=0.5)
ax.set_title('Inflation Differential: India minus US (%)', color='white')
ax.set_ylabel('Percentage points', color=C['soft'])
legend_patches = [
    mpatches.Patch(color=C['accent'], label='India inflation higher'),
    mpatches.Patch(color=C['ip'],     label='India inflation lower'),
]
ax.legend(handles=legend_patches, fontsize=8)

plt.tight_layout()
plt.savefig('data/chart_06_rupee_story.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()

In [ ]:
inr_2000 = float(usd_inr[usd_inr['date'].dt.year == 2000]['value'].mean())
inr_2024 = float(usd_inr[usd_inr['date'].dt.year == 2024]['value'].mean())
avg_diff = inf_diff.mean()

print(f"USD/INR 2000 avg: ₹{inr_2000:.1f}")
print(f"USD/INR 2024 avg: ₹{inr_2024:.1f}")
print(f"Avg inflation differential: {avg_diff:.1f} pp")

insight_box(f"""
The rupee has fallen from around ₹{inr_2000:.0f} to ₹{inr_2024:.0f} per dollar — a depreciation
of about {((inr_2024-inr_2000)/inr_2000*100):.0f}% over 24 years. This sounds alarming, but
economics explains most of it. Purchasing Power Parity theory predicts that a currency
should weaken at roughly the rate of its inflation advantage over the other country.
India's inflation has averaged {avg_diff:.1f} percentage points higher than the US over this
period — and the rupee's depreciation tracks this almost exactly. In other words, the
rupee hasn't become weaker in real terms; Indian goods have simply become more
expensive in rupee terms, so the exchange rate adjusts. The sharper drops in 2008,
2013 (the 'Taper Tantrum' when the US Fed hinted at rate hikes and foreign money fled
India), and 2020 were crisis-driven and temporary. The long-run trend is structural
and largely benign.
""".strip())

---
# Section 7 — Summary Dashboard
### The full India macro story in one view

In [ ]:
# ── Key statistics table ──────────────────────────────────────────
india_wb = wb[wb['country'] == 'India']

def stat(df, indicator, agg='mean'):
    vals = df[df['indicator'] == indicator]['value']
    return getattr(vals, agg)()

summary = pd.DataFrame([
    {'Metric': 'Avg GDP Growth (2000–2024)',      'India': f"{stat(india_wb,'gdp_growth_pct'):.1f}%",    'Context': 'Fastest among G20 economies'},
    {'Metric': 'Avg CPI Inflation (2001–2025)',   'India': f"{inflation['value'].mean():.1f}%",           'Context': 'Above RBI 4% target; improving post-2016'},
    {'Metric': 'GDP per Capita Growth (2000–now)','India': f"{multiple:.1f}x",                           'Context': f"${india_pc_2000:,.0f} → ${india_pc_last['value']:,.0f}"},
    {'Metric': 'Rupee Depreciation vs USD',       'India': f"{((inr_2024-inr_2000)/inr_2000*100):.0f}%",'Context': 'Largely explained by inflation differential'},
    {'Metric': 'Worst GDP shock (COVID 2020)',    'India': f"{india_2020:.1f}%",                         'Context': f'Rebounded {india_2021:.1f}% in 2021'},
    {'Metric': 'GFC impact (2009)',               'India': f"{india_2008:.1f}% growth",                  'Context': 'India barely contracted; banking sector insulated'},
])

print('\n' + '='*72)
print('  INDIA MACRO SUMMARY — KEY STATISTICS (2000–2025)')
print('='*72)
print(summary.to_string(index=False))
print('='*72)

In [ ]:
insight_box("""
India's macroeconomic journey from 2000 to 2025 is the story of a developing economy
gradually maturing its institutions. In the 2000s, India grew fast but with high
inflation and a reactive central bank. The 2010s brought institutional reform: formal
inflation targeting, a Monetary Policy Committee replacing one-person RBI decisions,
and the GST unifying the tax system. The 2020s began with COVID's brutal shock but
also revealed India's growing economic resilience — a stronger external balance,
record foreign exchange reserves, and a digital payments infrastructure that kept
commerce running even during lockdowns. The core tension going forward is the same
one every emerging economy faces: can India grow fast enough to employ its young
population while keeping inflation stable, the rupee credible, and inequality from
widening? The data so far suggests cautious optimism.
""".strip())

print('\n✅ Analysis complete. Charts saved to /data/')
print('   Next step → build the Streamlit dashboard (Phase 3)')